# Entrega 02 — Preparación de datos: Scania Component X

Objetivo: demostrar de forma sencilla y reproducible el flujo **Bronze → Silver → Gold**.

> Este notebook **no entrena ningún modelo**. Trabaja con una muestra de las lecturas operativas para que sea manejable en un equipo personal. Para procesar el dataset completo basta con eliminar o ampliar el límite `NROWS_OPERATIONAL`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw/scania")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

NROWS_OPERATIONAL = 250_000  # muestra técnica para la Entrega 02

ops_path = RAW_DIR / "train_operational_readouts.csv"
tte_path = RAW_DIR / "train_tte.csv"
spec_path = RAW_DIR / "train_specifications.csv"

for p in [ops_path, tte_path, spec_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"No se encuentra {p}. Descarga los ficheros desde "
            "https://researchdata.se/en/catalogue/dataset/2024-34/3"
        )


## 1. Bronze / Raw

Cargamos los archivos originales sin modificarlos.

In [2]:
ops_raw = pd.read_csv(ops_path, nrows=NROWS_OPERATIONAL)
tte_raw = pd.read_csv(tte_path)
spec_raw = pd.read_csv(spec_path)

print("Operational:", ops_raw.shape)
print("TTE:", tte_raw.shape)
print("Specifications:", spec_raw.shape)

display(ops_raw.head())
display(tte_raw.head())
display(spec_raw.head())


Operational: (250000, 107)
TTE: (23550, 3)
Specifications: (23550, 9)


,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,397_26,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,95728.0,15609.0,1984.0,8.0,784.0,150228.0,261904.0,93172.0,17874.0,452.0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,95729.0,15610.0,1984.0,8.0,784.0,150228.0,261905.0,93172.0,17874.0,452.0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,...,142900.0,19263.0,2441.0,12.0,1420.0,204832.0,313485.0,106464.0,19306.0,452.0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,...,150565.0,19832.0,2522.0,12.0,1444.0,211688.0,318901.0,107745.0,19406.0,453.0
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,NaN,NaN,NaN,NaN,...,155913.0,20573.0,2562.0,12.0,1445.0,213956.0,323997.0,109514.0,19535.0,454.0


,vehicle_id,length_of_study_time_step,in_study_repair
0,0,510.0,0
1,2,281.8,0
2,3,293.4,0
3,4,210.0,0
4,5,360.4,0


,vehicle_id,Spec_0,Spec_1,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7
0,0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0
1,2,Cat0,Cat1,Cat1,Cat0,Cat0,Cat0,Cat0,Cat1
2,3,Cat0,Cat1,Cat1,Cat1,Cat0,Cat0,Cat0,Cat1
3,4,Cat0,Cat0,Cat2,Cat1,Cat0,Cat0,Cat0,Cat1
4,5,Cat0,Cat2,Cat2,Cat0,Cat0,Cat0,Cat0,Cat1


## 2. Inspección inicial

In [3]:
def quality_summary(df, name):
    print(f"\n--- {name} ---")
    print("Filas / columnas:", df.shape)
    print("Duplicados exactos:", df.duplicated().sum())
    nulls = df.isna().sum().sort_values(ascending=False)
    print("Columnas con nulos:", (nulls > 0).sum())
    display(nulls[nulls > 0].head(15))

quality_summary(ops_raw, "Operational")
quality_summary(tte_raw, "TTE")
quality_summary(spec_raw, "Specifications")



--- Operational ---
Filas / columnas: (250000, 107)
Duplicados exactos: 0
Columnas con nulos: 104


291_1     2308
291_3     2308
291_0     2308
291_10    2308
291_9     2308
291_8     2308
291_2     2308
291_4     2308
291_5     2308
291_7     2308
291_6     2308
427_0     1946
370_0     1945
100_0     1945
459_10    1910
dtype: int64


--- TTE ---
Filas / columnas: (23550, 3)
Duplicados exactos: 0
Columnas con nulos: 0


Series([], dtype: int64)


--- Specifications ---
Filas / columnas: (23550, 9)
Duplicados exactos: 0
Columnas con nulos: 0


Series([], dtype: int64)

## 3. Silver

Aplicamos una limpieza mínima y transparente:

- tipos coherentes;
- eliminación de duplicados exactos;
- orden temporal;
- control de nulos;
- joins por `vehicle_id`.

No imputamos automáticamente las señales operativas porque primero debemos conocer la magnitud y el patrón de los nulos.


In [4]:
ops = ops_raw.copy()
tte = tte_raw.copy()
spec = spec_raw.copy()

# Limpieza de nombres por seguridad
for df in [ops, tte, spec]:
    df.columns = [str(c).strip() for c in df.columns]

required_ops = {"vehicle_id", "time_step"}
required_tte = {"vehicle_id", "length_of_study_time_step", "in_study_repair"}
required_spec = {"vehicle_id"}

assert required_ops.issubset(ops.columns), f"Faltan columnas en operational: {required_ops - set(ops.columns)}"
assert required_tte.issubset(tte.columns), f"Faltan columnas en TTE: {required_tte - set(tte.columns)}"
assert required_spec.issubset(spec.columns), "Falta vehicle_id en specifications"

# Tipos principales
ops["vehicle_id"] = pd.to_numeric(ops["vehicle_id"], errors="raise").astype("int64")
ops["time_step"] = pd.to_numeric(ops["time_step"], errors="coerce")

tte["vehicle_id"] = pd.to_numeric(tte["vehicle_id"], errors="raise").astype("int64")
tte["length_of_study_time_step"] = pd.to_numeric(
    tte["length_of_study_time_step"], errors="coerce"
)
tte["in_study_repair"] = pd.to_numeric(
    tte["in_study_repair"], errors="coerce"
).astype("Int64")

spec["vehicle_id"] = pd.to_numeric(spec["vehicle_id"], errors="raise").astype("int64")

# Duplicados exactos
ops = ops.drop_duplicates()
tte = tte.drop_duplicates()
spec = spec.drop_duplicates()

# Orden temporal
ops = ops.sort_values(["vehicle_id", "time_step"]).reset_index(drop=True)

print("Silver operational:", ops.shape)
print("Silver TTE:", tte.shape)
print("Silver specifications:", spec.shape)


Silver operational: (250000, 107)
Silver TTE: (23550, 3)
Silver specifications: (23550, 9)


In [5]:
# Comprobaciones sencillas
print("vehicle_id únicos en muestra operational:", ops["vehicle_id"].nunique())
print("vehicle_id únicos en TTE:", tte["vehicle_id"].nunique())
print("vehicle_id únicos en specifications:", spec["vehicle_id"].nunique())

print("\nDistribución del evento de reparación:")
display(tte["in_study_repair"].value_counts(dropna=False))


vehicle_id únicos en muestra operational: 4125
vehicle_id únicos en TTE: 23550
vehicle_id únicos en specifications: 23550

Distribución del evento de reparación:


in_study_repair
0    21278
1     2272
Name: count, dtype: Int64

## 4. Join Silver

Añadimos las especificaciones y la información del evento.  
Las variables de evento se conservarán como **objetivos/metadatos**, no como predictores.


In [6]:
silver = (
    ops
    .merge(spec, on="vehicle_id", how="left", validate="many_to_one")
    .merge(tte, on="vehicle_id", how="left", validate="many_to_one")
)

print("Silver unida:", silver.shape)

# Comprobación adicional de los joins:
# si ambos resultados son 0, todas las observaciones han encontrado
# sus especificaciones y su información TTE correctamente.
print(
    "Sin especificaciones tras el join:",
    silver["Spec_0"].isna().sum()
)
print(
    "Sin información TTE tras el join:",
    silver["in_study_repair"].isna().sum()
)

display(silver.head())


Silver unida: (250000, 117)
Sin especificaciones tras el join: 0
Sin información TTE tras el join: 0


,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,Spec_0,Spec_1,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7,length_of_study_time_step,in_study_repair
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0


## 5. Gold

Una fila representa el estado de un vehículo en un `time_step`.

Creamos dos campos de apoyo para la siguiente fase:

- `is_censored`: 1 cuando no se observó reparación durante el estudio.
- `time_to_event_from_row`: tiempo restante hasta la reparación únicamente cuando existe una reparación futura conocida.

Esto evita convertir automáticamente los casos censurados en “no fallo”.


In [7]:
gold = silver.copy()

gold["is_censored"] = (gold["in_study_repair"] == 0).astype("Int64")

future_event = (
    (gold["in_study_repair"] == 1)
    & gold["length_of_study_time_step"].notna()
    & gold["time_step"].notna()
    & (gold["length_of_study_time_step"] >= gold["time_step"])
)

gold["time_to_event_from_row"] = np.where(
    future_event,
    gold["length_of_study_time_step"] - gold["time_step"],
    np.nan
)

# Si apareciera alguna observación posterior al final del estudio, la excluimos.
valid_time = (
    gold["length_of_study_time_step"].isna()
    | gold["time_step"].isna()
    | (gold["time_step"] <= gold["length_of_study_time_step"])
)
gold = gold.loc[valid_time].reset_index(drop=True)

print("Gold:", gold.shape)
display(gold.head())


Gold: (250000, 119)


,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7,length_of_study_time_step,in_study_repair,is_censored,time_to_event_from_row
0,0,11.2,167985.0,10787.0,7413813.0,2296.0,4110.0,1296420.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,1,NaN
1,0,11.4,167985.0,10787.0,7413813.0,2296.0,4111.0,1302855.0,1628265.0,630345.0,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,1,NaN
2,0,19.6,331635.0,14525.0,13683604.0,2600.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,1,NaN
3,0,20.2,354975.0,15015.0,14540449.0,2616.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,1,NaN
4,0,21.0,365550.0,15295.0,14966985.0,2720.0,NaN,NaN,NaN,NaN,...,Cat0,Cat0,Cat0,Cat0,Cat0,Cat0,510.0,0,1,NaN


## 6. Guardar una muestra reproducible

Para el repositorio guardamos una muestra de hasta 10.000 filas de la capa Gold.

En lugar de utilizar simplemente las primeras filas del dataset, seleccionamos una **muestra aleatoria reproducible** mediante `random_state=42`. De este modo la muestra contiene observaciones más variadas y será exactamente la misma cada vez que se ejecute el notebook.


In [8]:
gold_sample_path = PROCESSED_DIR / "scania_gold_sample.csv"

n_sample = min(10_000, len(gold))

gold_sample = gold.sample(
    n=n_sample,
    random_state=42
)

gold_sample.to_csv(gold_sample_path, index=False)

print("Muestra guardada en:", gold_sample_path)
print("Filas:", len(gold_sample))
print("Vehículos únicos en la muestra:", gold_sample["vehicle_id"].nunique())
print("\nDistribución de in_study_repair en la muestra:")
display(gold_sample["in_study_repair"].value_counts(dropna=False))


Muestra guardada en: ..\data\processed\scania_gold_sample.csv
Filas: 10000
Vehículos únicos en la muestra: 3335

Distribución de in_study_repair en la muestra:


in_study_repair
0    8708
1    1292
Name: count, dtype: Int64

## Conclusión

El flujo deja trazado el paso desde los archivos originales hasta una tabla Gold utilizable en la siguiente fase. Todavía no se ha entrenado ningún modelo.

Antes del modelado habrá que decidir formalmente el objetivo (clasificación de riesgo, tiempo restante o supervivencia) y separar correctamente entrenamiento/validación sin utilizar información futura.
